### Pulling the newest version of repo

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'memory_harm_Shin-u'

%cd {PROJECT_DIR}
!git remote -v
!git branch --show-current
!git status

# Update with the newest repo
!git fetch origin
!git checkout shinu-experiment
!git pull --rebase origin shinu-experiment

### Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Set Project Working Directory

Set `PROJECT_DIR` once and reuse it throughout the notebook.


In [ ]:
import os
from pathlib import Path

project_dir = PROJECT_DIR
if not project_dir.exists():
    raise FileNotFoundError(
        f'Project directory not found: {project_dir}'
    )

os.chdir(project_dir)
print(f'Current working directory: {os.getcwd()}')


Expected project path: `/content/drive/MyDrive/memory_harm_Shinu`.


In [ ]:
# Verify the change in current working directory using a shell command
!pwd

### Installing Dependencies

**vLLM**:

vLLM is an inference engine used for loading the LLM models into GPU/CPU memory to perform efficent computations to get the output.

**unsloth**:

unsloth is a tool which provides the LLM models in a efficient way, a LLM model file has the weights, model architecture and computation code to perform computations using the weights and architecture. Unsloth performs optimization in providing the computations using Triton GPU language code (a bit fuzzy, written for understanding) which faster computations for training/fine-tuning and inference rather than using python code to communicate with GPU and perform the calculations. [Link](https://unsloth.ai/docs/blog/3x-faster-training-packing?ask=what+is+unsloth+used+for?+where+is+it+mentioned+regarding+the+uses+in+the+Unsloth+page)


**bitsandbites (quantization)**:

provides tools to reduce memory footprint (quantization) of LLM models via changing the numeric precision of the model weights invloved. High precision numbers require more memory and Low precision number require less memory.

In [ ]:
# Update pip first for cleaner installs
!pip install --upgrade pip

# Install Unsloth and vLLM (used Unsloth and bitsandbytes for \
# loading Unsloth_Qwen2.5_models)
!pip install unsloth vllm bitsandbytes

# 3. Install the specific requirements for memory_harm
!pip install -r requirements.txt

  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab-4.5.5-py3-none-any.whl.metadata (16 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 101.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 181.1 MB/s  0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [jupyter]


### Serving the LLM via vLLM

Model: Qwen2.5-7B-Instruct-bnb-4bit


Previously used, Qwen3.5-27B-FP8 model,but it took more than 5 mins to serve/load the model on the GPU, 5 min is the threshold I kept for timout.


The following code below:

```
!nohup python -m vllm.entrypoints.openai.api_server \
    --model $model \
    --served-model-name $model \
    --quantization "bitsandbytes" \
    --load-format "bitsandbytes" \
    --trust-remote-code \
    --port 8000 \
    --gpu-memory-utilization 0.9 > vllm.log 2>&1 &
```
creates a server using the Collab's GPU memory to store the LLM model and creates the endpoints (or uses server endpoints) compatible with OpenAI API endpoint connecting the API to server


In [ ]:
model = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

In [ ]:
import os
import subprocess
import time
import requests

# 1. Cleanup
!kill $(lsof -t -i:8000) >/dev/null 2>&1 || true
!rm -f nohup.out

# 2. Launch vLLM in the background
print("🚀 Launching vLLM server...")
!nohup python -m vllm.entrypoints.openai.api_server \
    --model $model \
    --served-model-name $model \
    --quantization "bitsandbytes" \
    --load-format "bitsandbytes" \
    --trust-remote-code \
    --port 8000 \
    --gpu-memory-utilization 0.9 > vllm.log 2>&1 &

# 3. Dynamic Health Check
print("⏳ Waiting for model to load into A100 VRAM...")
timeout = 300 # 5 minutes max
start_time = time.time()

while True:
    try:
        # Pinging the models endpoint is the best way to check if it's READY
        response = requests.get("http://localhost:8000/v1/models")
        if response.status_code == 200:
            print(f"\n✅ vLLM Server is UP and {model} model is loaded!")
            break
    except requests.exceptions.ConnectionError:
        pass

    if time.time() - start_time > timeout:
        print("\n❌ Error: Server timed out. Check 'vllm.log' for details.")
        break

    print(".", end="", flush=True)
    time.sleep(5)

🚀 Launching vLLM server...
⏳ Waiting for model to load into A100 VRAM...
..................................
✅ vLLM Server is UP and unsloth/Qwen2.5-7B-Instruct-bnb-4bit model is loaded!


## Skip the sections:


1.   Trial run of the LLM
2.   vLLM serving note


If the trial run of the LLM is successful. The trial run is for us (the developer), to understand whether the starter code is running as intended.





### Trial run of the LLM

We use vLLM's support on OpenAI SDK to run the LLM model to get the response for given prompt:

"San Francisco is a"

and the LLM should answer based on the prompt



In [ ]:
from openai import OpenAI

# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
response = client.chat.completions.create(model=model,
                                            messages=[
        {"role": "system", "content": "You are a helpful research assistant."},
        {"role": "user", "content": "San Francisco is a"}
                                            ],
                                            temperature=0.7,
                                            max_tokens=100
                                          )

# 3. Extract the message content
# Note the slightly different extraction path for Chat API
print("Qwen Response:", response.choices[0].message.content.strip())

Qwen Response: San Francisco is a city located in California, United States. It is known for its iconic landmarks such as the Golden Gate Bridge, Alcatraz Island, and Fisherman's Wharf. The city is also recognized for its hilly terrain, cable cars, and a diverse cultural scene. San Francisco is a major economic center and has a significant impact on technology, finance, and culture.


### vLLM Serving Note

We reading the file to see if vLLM has successfully loaded the model, if not the look into error and try to debug it.

In [7]:
!cat vllm.log

2026-03-04 22:32:55.524264: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 22:32:55.543768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772663575.567829    3089 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772663575.575134    3089 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772663575.592539    3089 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

### Running memory_harm, 'therapy' experiment

 With default paramters given in the repo, only the model is changed to
 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' for both user and assistant.

 The following experiment code works because we modified the src/utils.py, call_llm, call_llm_async function. The modification includes: using OpenAI sdk to point to our local (Collab GPU) server. Previously, we use OpenAI sdk to access their closed models which requires money (paid)



In [8]:
!pwd #Checking current working directory

/content/drive/MyDrive/Memory_Harm/memory_harm_repo/memory_harm_laxman


In [ ]:
!ls #Checking the repo files

configs		   project_plan.md  requirements.txt   STATUS.md
data		   QUICKSTART.md    roadmap.md	       tests
implementation.md  README.md	    run_experiment.py  vllm.log
latex		   reports	    src


In [9]:
import requests
response = requests.get("http://localhost:8000/v1/models")
print(response.json()) #Checking whether our server has the LLM model loaded

{'object': 'list', 'data': [{'id': 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit', 'object': 'model', 'created': 1772665410, 'owned_by': 'vllm', 'root': 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-bb82a2aea02391ae', 'object': 'model_permission', 'created': 1772665410, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


The parameters:

*   seed
*   assistant_model
*   user_model

are not changeable in the run experiment script below, should change in the exp.yaml file inside the config directory

**After running the experiment, 'data' directory will be created which has 'logs' directory containing the results as .log file**. Use dashboard.ipynb inside the reports/ directory to look into the results




### "none" memory mode

In [ ]:
# Run a single experiment
!python -m src.sim --config configs/exp.yaml\
  --memory_mode "none" \
  --scenario "therapy" \
  --episodes 50 \
  --conversations 5 \
  --steps 6 \
  --max_concurrent 20


### "summary" memory mode

In [ ]:
# Run a single experiment
!python -m src.sim --config configs/exp.yaml\
  --memory_mode "summary" \
  --scenario "therapy" \
  --episodes 50 \
  --conversations 5 \
  --steps 6 \
  --max_concurrent 20


### "full_context" memory mode

In [ ]:
# Run a single experiment
!python -m src.sim --config configs/exp.yaml\
  --memory_mode "full_context" \
  --scenario "therapy" \
  --episodes 50 \
  --conversations 5 \
  --steps 6 \
  --max_concurrent 20


### Post-Run Workflow

This section records run metadata to Drive, analyzes logs, and checkpoints the notebook.


In [ ]:
from src.colab_workflow import (
    mount_drive_if_needed,
    DriveLayout,
    save_run_manifest,
    save_notebook_checkpoint,
)

mount_drive_if_needed()
layout = DriveLayout.from_drive(project_name=PROJECT_DIR.name, drive_root=str(DRIVE_ROOT))
layout.ensure_dirs()
print(f'Log dir: {layout.log_dir}')
print(f'Runs dir: {layout.runs_dir}')


In [ ]:
# 1) Analyze latest logs saved on Drive
!python reports/analyze.py --log_dir {layout.log_dir}

# 2) Save run metadata (edit values to match your run)
manifest_path = save_run_manifest(
    layout,
    note='therapy run in Colab',
    extra={'scenario': 'therapy', 'episodes': 50, 'conversations': 5, 'steps': 6},
)
print(f'Saved run manifest: {manifest_path}')

# 3) Save notebook checkpoint (for Drive-backed notebook)
save_notebook_checkpoint()
print('Notebook checkpoint requested.')

# Optional code-only commit workflow (do this only when source code changed):
# !git status
# !git add src configs reports QUICKSTART.md
# !git commit -m "Update experiment code"
# !git push
